In [1]:

import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, Bidirectional, LSTM
from tensorflow.keras.callbacks import EarlyStopping


In [4]:
data = pd.read_csv(
    "/content/IMDB Dataset (1).csv",
    engine='python',          # more flexible parser
    encoding='utf-8',         # handle text properly
    on_bad_lines='skip'       # skip corrupted rows
)

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data["review"] = data["review"].apply(clean_text)
data["sentiment"] = data["sentiment"].map({"positive":1, "negative":0})

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    data["review"], data["sentiment"],
    test_size=0.2,
    random_state=42
)


In [7]:
vocab_size = 40000
max_len = 200

In [8]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train) ##word to index mapping

In [9]:
X_train_seq = tokenizer.texts_to_sequences(X_train) ##sentence to sequence of numbers
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [10]:
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")


In [11]:
model = Sequential([
    Embedding(vocab_size, 128),

    Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)),

    Dense(64, activation='relu'),
    Dropout(0.6),

    Dense(1, activation='sigmoid')
])

In [12]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=6,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 84s 636ms/step - accuracy: 0.6573 - loss: 0.6021 - val_accuracy: 0.7739 - val_loss: 0.4731
Epoch 2/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 77s 613ms/step - accuracy: 0.8197 - loss: 0.4336 - val_accuracy: 0.8174 - val_loss: 0.4132
Epoch 3/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 83s 623ms/step - accuracy: 0.8682 - loss: 0.3390 - val_accuracy: 0.8304 - val_loss: 0.3960
Epoch 4/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 621ms/step - accuracy: 0.8966 - loss: 0.2830 - val_accuracy: 0.8155 - val_loss: 0.4251
Epoch 5/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 620ms/step - accuracy: 0.9176 - loss: 0.2342 - val_accuracy: 0.8160 - val_loss: 0.4393
Epoch 6/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 83s 624ms/step - accuracy: 0.9300 - loss: 0.2031 - val_accuracy: 0.8285 - val_loss: 0.4386


In [15]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 41ms/step - accuracy: 0.8340 - loss: 0.3857

Test Accuracy: 83.40%


In [16]:
def predict_review(review):
    review = clean_text(review)
    seq = tokenizer.texts_to_sequences([review])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')
    pred = model.predict(pad)[0][0]
    return "Positive" if pred > 0.5 else "Negative"

In [17]:
print("What a boring day!", predict_review("What a boring day!"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 767ms/step
What a boring day! Negative


In [18]:
print("I absolutely loved this movie, it was fantastic!", predict_review("I absolutely loved this movie, it was fantastic!"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
I absolutely loved this movie, it was fantastic! Positive
